# Visualisation

**Note:** To run this notebook, users should have ipykernel and matplotlib
installed. This can be done by running 
```bash
uv sync --extra examples
```
in the terminal.

In [ ]:
from train import load_data, downsample, load_model

inputs, outputs, metadata = load_data("data/nu0p2_res2048.h5")

print("Original data:")
print("    metadata:", metadata)
print("    inputs.shape:", inputs.shape)
print("    outputs.shape:", outputs.shape)

inputs, outputs = downsample(inputs, outputs, 1024)

print("Downsampled data:")
print("    inputs.shape:", inputs.shape)
print("    outputs.shape:", outputs.shape)

model, _ = load_model("models/burgers1d_fno_64.eqx")

print(model)

In [ ]:
import jax.numpy as jnp

num_samples = inputs.shape[0]
num_train = 2048
num_test = 512
assert num_train + num_test == num_samples

inputs_jax = jnp.asarray(inputs)
outputs_jax = jnp.asarray(outputs)

test_data = {
    "input": inputs_jax[num_train:],
    "output": outputs_jax[num_train:],
}

In [ ]:
import jax
import matplotlib.pyplot as plt

key = jax.random.key(0)
num_examples = 3

key, subkey = jax.random.split(key)
random_inds = jax.random.choice(subkey, num_test, (num_examples,))

example_inputs = test_data["input"][random_inds]
example_outputs = test_data["output"][random_inds]
predictions = jax.vmap(model)(example_inputs)

fig, axes = plt.subplots(
    2, num_examples, figsize=(16, 9), layout="tight", sharex=True
)

for i in range(num_examples):
    idx = random_inds[i]
    x = example_inputs[i, :, 0]
    u0 = example_inputs[i, :, 1]
    v_true = example_outputs[i, ...]
    v_pred = predictions[i, ...]

    axes[0, i].plot(x, u0)
    axes[0, i].set_ylabel("$u_0(x)$", fontsize=10)
    axes[0, i].set_title(f"Sample {idx}", fontsize=11)

    axes[1, i].plot(x, v_pred, "-", label="Prediction")
    axes[1, i].plot(x, v_true, "--", label="Ground truth")
    axes[1, i].set_xlabel("$x$", fontsize=10)
    axes[1, i].set_ylabel("$u(x, t=1)$", fontsize=10)
    axes[1, i].legend(fontsize=10)

plt.show()